# Task5 — Five-bag density, alert expansion, and conditional permutations

สามการทดลองแยกกัน: (1) ขยาย original ranking จาก 2,046 เป็น 2,250 alerts ตามที่ผู้ใช้เลือก (2) เพิ่ม unconditional density เป็น 5 bags โดยเทียบที่ 2,046 alerts (3) เปลี่ยนเฉพาะ HGB null generation เป็น conditional permutation จำนวน 5 bags โดยเทียบที่ 2,046 alerts

Run All default อ่าน fitted results และตรวจ submission files ไม่ retrain และไม่ upload Kaggle ไม่มี measured anomaly F3

## 1. Setup

In [1]:
from pathlib import Path
import sys
import json
import pandas as pd

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "Task5/models/density_context_followup.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Open from within CPE342-Kaggle")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from Task5.models import density_context_followup as experiment
from Task5.models import density_context_exports as exports
OUT = experiment.OUT
RUN_TRAINING = False
RUN_FULL_MODEL_VERIFICATION = False
print(OUT)

/Users/meng/dev/MyUniversity/CPE342-MachineLearning/CPE342-Kaggle/Task5/models/results/density_context5_v1


## 2. Protocol and saved configuration

เก็บ geometry, temporal-shift posterior, marginal-mode posterior, background และ 5 folds เดิม Unconditional branch reuse 15 models เดิมและเพิ่ม bags4–5 อีก 10 fits Conditional branch ฝึก 25 fits ใหม่โดยเปลี่ยนเฉพาะวิธีสร้าง null data ใน train/evaluation

Conditional strata มาจาก training-background quartiles ของ account_age_days และ mean activity_level periods1–3 มี missing bins แยก ไม่ใช้ period4 ในการกำหนด strata แต่ shared background/geometry/calibration ยังคงเป็น transductive

In [2]:
experiment.CONFIG

{'folds': 5,
 'bags': 5,
 'projections': 140,
 'iterations': 160,
 'training_null_repeats': 3,
 'evaluation_null_repeats': 2,
 'seed': 342,
 'threads': 4,
 'max_bins': 63,
 'learning_rate': 0.07,
 'max_depth': 5,
 'l2_regularization': 2.0,
 'context_features': ['account_age_days', 'activity_history_mean'],
 'context_quantile_cuts': [0.25, 0.5, 0.75],
 'matched_alert_budget': 2046,
 'scope': 'Context rules fit only on training background. Shared geometry/calibration are transductive.'}

## 3. Train only when requested

แต่ละ completed model save atomic เพื่อ resume ถ้า code/config/input hashes เปลี่ยนให้ใช้ผล version ใหม่ ไม่ bypass cache guard

In [3]:
if RUN_TRAINING:
    experiment.initialize()
    experiment.train("unconditional5")
    experiment.train("conditional5")
    experiment.report()
    experiment.verify()
summary = json.loads((OUT / "summary.json").read_text())
display(pd.DataFrame(summary["comparisons"]))

,mode,policy,alerts,common_with_best,added,removed,jaccard_with_best,dependence_posterior_above_half,measured_f3
0,unconditional5,auto_prediction,2034,2020,14,26,0.980583,241,None
1,unconditional5,fixed2046_prediction,2046,2022,24,24,0.976812,241,None
2,conditional5,auto_prediction,2024,2010,14,36,0.975728,235,None
3,conditional5,fixed2046_prediction,2046,2010,36,36,0.965418,235,None


## 4. Auxiliary diagnostics

AUC เป็นการแยก real/background กับ generated null ไม่ใช่ fraud AUC และ conditional กับ unconditional null เป็นโจทย์คนละแบบ จึงใช้ค่าที่สูงขึ้นอ้างว่า anomaly model ดีขึ้นไม่ได้ Auto-policy count แสดงเพื่อดู sensitivity แต่ submission ของสองโมเดลใหม่คง 2,046 alerts เพื่อแยก ranking effect

In [4]:
logs = []
for mode in ["unconditional5", "conditional5"]:
    records = json.loads((OUT / f"{mode}_training_log.json").read_text())
    auc = [r["record"].get("auxiliary_auc", r["record"].get("real_vs_shuffled_holdout_auc")) for r in records]
    logs.append({"mode": mode, "fold_models": len(records), "new_fits": sum(not r["reused_from_best"] for r in records), "auxiliary_auc_mean": sum(auc)/len(auc), "new_fit_seconds": sum(r["record"]["seconds"] for r in records if not r["reused_from_best"])})
display(pd.DataFrame(logs))
print("Measured anomaly F3: unavailable")

,mode,fold_models,new_fits,auxiliary_auc_mean,new_fit_seconds
0,unconditional5,25,10,0.509215,13.092655
1,conditional5,25,25,0.511781,34.064513


Measured anomaly F3: unavailable


## 5. Original-ranking expansion and matched-budget candidates

Expanded candidate คง ranking ของรุ่น .81771 ทุกแถว ไม่เอาโมเดลใหม่มาปน และ original alerts ทั้ง 2,046 ต้องยังอยู่ ส่วน paired conditional effect เปรียบเทียบกับ unconditional 5 bags ไม่ใช่ 3 bags เพื่อไม่ปนผล bag count

In [5]:
candidate_table, paired_conditional_effect = exports.prepare()
display(candidate_table[["name", "alerts", "retained_best_alerts", "added", "removed", "changed_task5_rows"]])
paired_conditional_effect

,name,alerts,retained_best_alerts,added,removed,changed_task5_rows
0,density_expand2250_v1,2250,2046,204,0,204
1,density5bags_fixed2046_v1,2046,2022,24,24,48
2,conditional5bags_fixed2046_v1,2046,2010,36,36,72


{'comparison': 'conditional5 vs unconditional5 at 2046 alerts',
 'added': 23,
 'removed': 23,
 'jaccard': 0.9777670372160464,
 'measured_anomaly_f3': None}

## 6. Verify fitted artifacts

ตรวจ 50 fold bundles รวม 15 ที่ reuse คำนวณ held-out real/null logits ซ้ำ ตรวจ conditional permutation ไม่ข้าม strata และรักษา per-column marginals/NaNs ภายใน strata ตรวจ original 3-bag ranking ได้ตรงทุกค่า และ shift/marginal-mode scores ไม่เปลี่ยน

In [6]:
if RUN_FULL_MODEL_VERIFICATION:
    experiment.verify()
json.loads((OUT / "verification.json").read_text())

{'fold_bundle_checks': 50,
 'heldout_real_predictions_recomputed': 258890,
 'heldout_null_predictions_recomputed': 481960,
 'no_scored_row_in_its_fit': True,
 'conditional_context_rules_fit_on_training_background_only': True,
 'conditional_null_within_stratum_marginals_and_missingness_preserved': True,
 'unchanged_shift_and_island_views_verified': True,
 'original_three_bag_ranking_exactly_reproduced': True,
 'anomaly_accuracy_verified': False}

## 7. Prepared CSVs and independent checks

สร้างใหม่เฉพาะไฟล์ที่ยังไม่มี ถ้ามีอยู่แล้วตรวจโดยไม่ overwrite ทุก candidate คง IDs/Task1–4 จาก density_auto submission เดิม ไม่จำเป็นต้องส่งทั้งสามไฟล์ กำหนดสมมติฐานและลำดับทดสอบก่อนใช้ quota

In [7]:
checks = exports.export_all()
display(pd.DataFrame(checks))
for item in checks:
    print(item["output"])

,output,output_sha256,rows,alerts,ids_and_tasks1_to4_literal_fields_preserved,task5_decisions_independently_checked
0,/Users/meng/dev/MyUniversity/CPE342-MachineLea...,e7e4881ed008ec64be130339910beefa826854e8d0e73f...,25889,2250,True,True
1,/Users/meng/dev/MyUniversity/CPE342-MachineLea...,fbc4dfb2194cd34c571544a5235c85e5a2855c90c8def2...,25889,2046,True,True
2,/Users/meng/dev/MyUniversity/CPE342-MachineLea...,def643c70af1219e7a565fcd5bf4652b334b15c20596f8...,25889,2046,True,True


/Users/meng/dev/MyUniversity/CPE342-MachineLearning/CPE342-Kaggle/submission/kaggle_t12345_t1_slow800_bagging_v4_t3_conditional_quantile_v1_t4_resnet18_cosine18_tta_v1_t5_density_expand2250_v1.csv
/Users/meng/dev/MyUniversity/CPE342-MachineLearning/CPE342-Kaggle/submission/kaggle_t12345_t1_slow800_bagging_v4_t3_conditional_quantile_v1_t4_resnet18_cosine18_tta_v1_t5_density5bags_fixed2046_v1.csv
/Users/meng/dev/MyUniversity/CPE342-MachineLearning/CPE342-Kaggle/submission/kaggle_t12345_t1_slow800_bagging_v4_t3_conditional_quantile_v1_t4_resnet18_cosine18_tta_v1_t5_conditional5bags_fixed2046_v1.csv


## 8. Status against the provided hints

| Hint | สถานะ |
| --- | --- |
| Isolation Forest / One-Class SVM | งานเดิมมี fitted detector families และ OCSVM submissions บันทึกไว้; ยังไม่ได้ fit ใหม่ในรอบนี้ |
| Autoencoder | ยังไม่ได้ฝึก |
| Unsupervised ensemble | เดิมมี OCSVM/GMM rank blend และ consensus/union; ตัวหลักรวมสาม views และ HGB bagging; residual fusion95/5 ส่งแล้วแพ้ |
| Soft / hard voting | Rank fusion ไม่เท่ากับ soft voting จาก calibrated probabilities; ยังไม่มี voting experiment ใหม่ร่วมกับ density_auto ในรอบนี้ |
| Tune contamination / threshold | ตัวหลักใช้ model-based auto policy; รอบนี้แยกทดลองเพิ่ม original ranking เป็น 2,250 alerts |

สไลด์ระบุ 32 predictors แต่ dataset ปัจจุบันมี 122 predictors จึงยึด raw data และ current schema ไม่ hardcode จำนวนตามภาพ ความเห็นตรงกันของ detectors ไม่ใช่ accuracy และ F3 ไม่ได้วัด recall อย่างเดียว